# Entrenar CNN sin usar terminal

Esta notebook permite ejecutar `src/train.py` desde Jupyter o Google Colab. Antes de correrla, asegúrate de que el proyecto tenga esta estructura:

```text
DL6/
├── requirements.txt
├── src/
│   ├── train.py
│   ├── eval.py
│   ├── predict.py
│   └── config.yaml
└── dataset/
    ├── train/
    ├── val/
    └── test/images/
```

Si estás en Colab, sube o monta la carpeta completa del proyecto y ajusta `PROJECT_DIR` en la siguiente celda.

In [ ]:
from pathlib import Path
import os
import sys

# Cambia esta ruta si estás usando Colab y tu proyecto está en otra carpeta.
PROJECT_DIR = Path.cwd()

# Ejemplo Colab con Drive montado:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/DL6')

os.chdir(PROJECT_DIR)
print('Directorio de trabajo:', Path.cwd())
print('Python:', sys.version)

## Instalar dependencias

En Colab puede tardar unos minutos. Si ya tienes las dependencias instaladas, puedes saltar esta celda.

In [ ]:
%pip install -q -r requirements.txt

## Revisar archivos y carpetas

In [ ]:
required_paths = [
    Path('requirements.txt'),
    Path('src/train.py'),
    Path('src/eval.py'),
    Path('src/predict.py'),
    Path('src/config.yaml'),
    Path('dataset/train'),
    Path('dataset/val'),
    Path('dataset/test/images'),
]

for path in required_paths:
    print(('OK   ' if path.exists() else 'FALTA'), path)

## Entrenar

El comando usa explícitamente `src/config.yaml`.

In [ ]:
!python src/train.py --config src/config.yaml

## Elegir checkpoint

Después de entrenar, esta celda busca el checkpoint `best.pt` más reciente.

In [ ]:
checkpoints = sorted(Path('output').glob('*/checkpoints/best.pt'), key=lambda p: p.stat().st_mtime)
if not checkpoints:
    raise FileNotFoundError('No se encontró ningún checkpoint best.pt en output/*/checkpoints/')

CHECKPOINT = checkpoints[-1]
print('Checkpoint:', CHECKPOINT)

## Evaluar validación etiquetada

`src/eval.py` calcula métricas usando `dataset/val/`.

In [ ]:
!python src/eval.py --config src/config.yaml --checkpoint "$CHECKPOINT"

## Generar predictions.csv

`src/predict.py` usa `dataset/test/images/`, que no tiene etiquetas. El archivo `predictions.csv` es el que se entrega al leaderboard.

In [ ]:
!python src/predict.py --config src/config.yaml --checkpoint "$CHECKPOINT" --output predictions.csv

In [ ]:
import csv
from itertools import islice

with open('predictions.csv', newline='') as f:
    for row in islice(csv.reader(f), 6):
        print(row)